# 🫀 실험 22 — **경색을 흉내내는 것들**: 위양성을 감별질환별로 쪼갠다

**MedKOS / `notebooks/exp22_mimic_false_positives.ipynb`** · 퀘스트 `ailab-2026-0015`
**학습 0회** · 실험16~19 저장 arm 만 읽는다 · 다운로드 없음 · **~5분**

---

## 왜 이걸 해야 하나

지금까지 AUROC 의 음성군은 **전량 21,799건**이었다(NORM 만이 아니다 — 확인했다).
그러니 `CLBBB`·`LVH`·`WPW` 는 **이미 음성군 안에 들어 있다.**

**그런데 희석돼 있다.** `CLBBB` 는 전체의 2.5% 남짓이라, 거기서 위양성이 100% 여도
전체 AUROC 는 소수점 셋째 자리밖에 안 움직인다. **우리는 감별질환별 위양성을 분해해
본 적이 한 번도 없다.**

임상에서 특이도를 무너뜨리는 것은 **정상 심전도가 아니라 이들**이다:

| 기전 | 소견 | 무엇을 흉내내나 |
|---|---|---|
| **병적 Q파 · QS 패턴** | `CLBBB` `WPW`(음성 델타파) | 경색의 Q파 |
| **R파 진행 불량** | `LVH` `CLBBB` `SEHYP` | **전벽/전중격 경색** |
| ST 변화 | `LVH`(strain) `STTC` | 손상전류 |

교과서가 정리하는 그대로다 — WPW 조기흥분·비후성심근증·LVH·RVH·좌각차단은
병적 Q파나 QS 패턴, R파 진행 불량을 만들어 **경색을 흉내낸다.**

## 이 실험이 정말로 묻는 것 — 배포 후보가 살아남나

`{I,II,V2,V5}`(5전극)는 AUROC 로는 12유도와 같았다(최악 D +0.0076).
그런데 **LBBB 를 전중격 경색과 구별하려면 V1–V3 형태를 봐야 한다.**
V2 하나만 있는 구성이 그걸 할 수 있는가?

> **AUROC 가 같다고 감별 능력이 같다는 보장이 없다.**
> 이게 확인 안 되면 "5전극 = 12유도" 라는 헤드라인에 단서를 달아야 한다.

## 무엇을 계산하나

부위 `s`, 감별질환군 `G` 에 대해

```
FPR(s, G) = P( s 경보 | s 음성 · G 소견 있음 · MI 코드 전혀 없음 )
배수(s, G) = FPR(s, G) / FPR(s, NORM)
```

- **`MI 코드 전혀 없음`** 이 핵심이다. LBBB + 하벽경색이 같이 있는 레코드는
  깨끗한 모방 검정이 아니므로 뺀다.
- 동작점은 **실험14 규약대로 교차적합**(검증 겹에서 민감도 0.90 → 테스트 겹 적용).
- **단일단계와 2단계를 둘 다** 계산한다 → **실험19 결과에 의존하지 않는다.**

## 사전등록 (결과 보기 전에 고정)

판정은 시드 t-CI. 소집단은 레코드 부트스트랩 CI 도 함께 내고 **넓은 쪽**을 쓴다.

| | 예측 | 성격 |
|---|---|---|
| **G0** | 코드 어휘 확인 · arm 정합성 · 군별 n ≥ 50 | 전제 |
| **P-1 ★★** | `CLBBB` 군의 **`ASMI`** 위양성 **오즈비 ≥ 2** (12유도) | 주가설·**단일 검정**. LBBB 는 전중격 경색의 고전적 모방이다 |
| **P-2 ★** | 모방군의 **기전이 예측하는 부위에서** macro OR ≥ 2 | 기전이 일반화되나 |
| **P-3 ★★ (특이성 대조)** | 모방군 OR − 전도장애군(`CRBBB`·`IRBBB`·`IVCD`) OR **≥ 0.5** | **아무 이상에나 반응하는 게 아니라 Q파 기전에서 크다** |
| **P-4 ★★ (배포 후보 검증)** | `{I,II,V2,V5}` 의 모방군 OR ≤ `{12}` OR + 0.5 | **유도를 줄여도 감별 능력이 안 무너지나** |
| **P-5** | 2단계 동작점이 단일단계보다 모방군 위양성률을 낮춘다 | 실험19 후속(결과와 무관하게 계산) |

> **P-3 이 없으면 P-1·P-2 는 아무 의미가 없다.** "비정상 심전도는 다 위양성이 많다" 가
> 사소한 대안 설명이기 때문이다. 전도장애군은 **QRS 는 넓지만 Q파 기전은 없는** 대조군이다.
> (`IRBBB` 는 실험13b 에서 D 0.130 vs `CRBBB` 0.003 으로 40배 갈렸던 그 쌍이다.)

> ⚠️ **macro 를 7부위 전체로 잡지 않는다.** 모방은 **부위 특이적**이다 —
> `CLBBB`·`LVH` 는 R파 진행 불량으로 **전벽**(ASMI·AMI)을, `WPW` 는 음성 델타파로
> **하벽**(IMI)을 흉내낸다. 7부위 평균을 내면 그 기전이 희석된다(픽스처: 모방 OR
> **3.80 → macro 1.42**). 실험17 P-4·P-5 가 정확히 이렇게 무너졌다.
> → **모방별 표적 부위를 사전지정**하고 거기서만 평균한다. 대조군(전도장애)은 예측
> 부위가 없으므로 **같은 표적 합집합**에서 재 공정하게 맞춘다. 7부위 표는 탐색으로 낸다.

> ⚠️ **왜 위험비가 아니라 오즈비인가.** 위험비 `FPR(G)/FPR(NORM)` 는 **천장이
> `1/FPR(NORM)`** 이다. 우리 경보율이 0.2~0.5 라 `FPR(NORM)=0.5` 인 부위에서는
> 아무리 나빠도 위험비가 2 를 못 넘는다 — 픽스처에서 기전을 심었는데 **1.56 에서
> 막혔다.** 오즈비는 천장이 없다. 표에는 **OR·위험비·위험차 셋 다** 내고 판정은 OR 로 한다.

> ⚠️ **P-3 의 문턱은 0 이 아니라 0.5 다.** 표본이 21,799건이라 시드 SD 가 극히 작아
> **+0.01 짜리 차이도 CI 가 0 을 벗어난다**(픽스처에서 귀무가 통과했다).
> 통계적 유의가 아니라 **최소 유의미 차이**를 사전에 못 박는다.

> ⚠️ **분모 가드**: `NORM` 의 기대 위양성이 5건 미만인 칸은 배수가 폭발하므로
> **배수를 내지 않고 위험차(`FPR(군) − FPR(NORM)`)로만 읽는다.** 픽스처에서 분모가
> 0에 가까울 때 배수가 10⁷ 까지 튀는 것을 확인하고 넣었다. 표에는 **배수와 위험차를
> 나란히** 낸다.

> ⚠️ **`WPW` 는 n≈80 으로 작다.** 소집단은 CI 가 넓게 나올 것이고, `n < 50` 이면
> 채점에서 자동 제외하고 표에만 남긴다. **IPLMI 에서 겪은 실수를 반복하지 않는다.**

## 이 실험이 무엇을 정하나

- `P-1`·`P-2`·`P-3` ✅ → **어느 감별질환부터 라벨 트리에 넣어야 하는지가 추측이 아니라
  측정으로 정해진다.** 2단계 라벨 공사의 입력이 된다.
- `P-4` ❌ → **"5전극 = 12유도" 헤드라인에 단서를 단다**: AUROC 는 같지만 감별에서
  진다. 배포 사양을 다시 본다.
- 전부 ❌ → 감별질환은 지금 구조가 이미 잘 처리하고 있다. 라벨 트리 확장 우선순위를 낮춘다.


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · "
      "assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정 + 실험16~19 arm 연결 (학습 0회)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~19 와 한 글자도 달라선 안 되는 블록
K_FOLD, SEED0, NMIN = 5, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
# ★★ 여기까지

CFGS = ["12", "I+II+V2+V5", "I+II+V1"]     # 12유도 = 기준 · 5전극 배포후보 · 4전극
REF_CFG = "12"
DEPLOY  = "I+II+V2+V5"
SENS    = 0.90        # 부위 동작점 (실험14~18 과 동일)
SENS_L1 = 0.95        # 2단계 1차 게이트
GMIN    = 50          # 군 최소 표본 — 미만은 채점 제외(표에만)
MIN_FP  = 5           # ★ NORM 위양성 기대건수 하한. 분모가 0에 가까우면 배수가 폭발한다
                      #   (픽스처에서 FPR(NORM)≈0 일 때 배수가 1e7 까지 튀었다)
BOOT    = 2000

# ── 감별질환군 (PTB-XL **scp 코드** 층위. subclass 아님 — 실험15 이후 일관)
#    ★ 기전으로 묶는다. P-3 의 대조군이 성립하려면 이 구분이 핵심이다.
MIMIC_Q = ["CLBBB", "LVH", "WPW"]          # 병적 Q파·QS·R파 진행 불량을 만드는 군
CONDUCT = ["CRBBB", "IRBBB", "IVCD"]       # QRS 는 넓지만 Q파 기전은 없다(특이성 대조)
EXTRA   = ["ILBBB", "RVH", "SEHYP", "LAFB", "LPFB", "1AVB", "STTC", "NST_"]  # 탐색
P1_MIMIC, P1_SITE = "CLBBB", "ASMI"        # ★ 주가설 — 단일 검정

# ★★ 모방 → **기전이 예측하는 부위** (사전지정)
#    7부위 전체로 macro 를 잡으면 부위 특이적 기전이 희석된다 — 실험17 P-4·P-5 가
#    정확히 그렇게 무너졌고, 픽스처에서 재발을 확인했다(모방 OR 3.80 → macro 1.42).
MIMIC_SITES = {"CLBBB": ["ASMI", "AMI"],   # QS/R파 진행 불량 → 전벽
               "LVH":   ["ASMI", "AMI"],   # R파 진행 불량 → 전벽
               "WPW":   ["IMI"]}           # 음성 델타파 → 하벽
TARGET_SITES = ["ASMI", "AMI", "IMI"]      # 대조군은 예측 부위가 없으므로 이 합집합에서
OR_THR   = 2.0        # P-1·P-2 — **오즈비** 문턱
P3_MARGIN = 0.5       # P-3 — 모방군 OR 이 전도장애군 OR 보다 이만큼은 커야 한다
P4_SLACK  = 0.5       # P-4 — 배포 후보가 12유도보다 이만큼 넘게 나쁘면 안 된다
# ★ 왜 위험비가 아니라 오즈비인가:
#   위험비 FPR(G)/FPR(NORM) 는 **천장이 1/FPR(NORM)** 이다. 우리 경보율이 0.2~0.5 라
#   FPR(NORM)=0.5 인 부위에서는 아무리 나빠도 위험비가 2 를 못 넘는다(픽스처에서 확인:
#   기전을 심었는데 1.56 에서 막혔다). 오즈비는 천장이 없다. 표에는 셋 다 낸다.

# 깨끗한 모방 검정을 위해 **MI 계열 코드가 하나라도 있으면 제외**한다
MI_LIKE = SITE_CANDIDATES + ["INJAS", "INJAL", "INJIN", "INJLA", "INJIL"]

CONFIG = dict(exp="exp22_mimic_false_positives", quest="ailab-2026-0015",
              parent_exp=["exp16_four_cfg", "exp17_wearable",
                          "exp18_confirm", "exp19_two_stage"],
              purpose=("경색을 흉내내는 소견별로 위양성을 쪼갠다 — 감별질환은 음성군에 "
                       "섞여 있을 뿐 희석돼 있어 전체 AUROC 로는 안 보인다"),
              training=0, configs=CFGS, deploy=DEPLOY, ref=REF_CFG,
              mimic_q=MIMIC_Q, conduct=CONDUCT, extra=EXTRA,
              mimic_sites=MIMIC_SITES, target_sites=TARGET_SITES,
              p1=dict(mimic=P1_MIMIC, site=P1_SITE, or_thr=OR_THR),
              exclude="MI 계열 코드가 하나라도 있으면 제외(깨끗한 모방만)",
              operating_point=f"검증 겹에서 민감도 {SENS} → 테스트 겹 적용(교차적합)",
              judged_on="시드 t-CI + 레코드 부트스트랩 — 넓은 쪽을 쓴다",
              predictions={
                  "P-1": f"{P1_MIMIC} 군의 {P1_SITE} 위양성 **오즈비** >= {OR_THR} (12유도, 단일 검정)",
                  "P-2": f"모방군 {MIMIC_Q} macro 오즈비 >= {OR_THR}",
                  "P-3": f"모방군 OR - 전도장애군 {CONDUCT} OR >= {P3_MARGIN} (특이성 대조)",
                  "P-4": f"{DEPLOY} 의 모방군 OR <= {REF_CFG} OR + {P4_SLACK}",
                  "P-5": "2단계 동작점이 단일단계보다 모방군 위양성률을 낮춘다"},
              caveat=(f"군 n < {GMIN} 은 채점 제외하고 표에만 남긴다 — IPLMI(n=51) 에서 "
                      "겪은 '측정 불가를 결과로 취급' 실수를 반복하지 않는다"),
              k_fold=K_FOLD, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp22_mimic_fp", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") in ("exp16_four_cfg", "exp17_wearable",
                           "exp18_confirm", "exp19_two_stage") \
            and os.path.isdir(r.get("dir", "")):
        DIRS[r["exp_id"]] = r["dir"]
if not DIRS:
    raise RuntimeError("registry.jsonl 에서 실험16~19 를 하나도 못 찾았습니다")
PARENTS = [DIRS[k] for k in ("exp19_two_stage", "exp18_confirm",
                             "exp17_wearable", "exp16_four_cfg") if k in DIRS]
for k in ("exp16_four_cfg", "exp17_wearable", "exp18_confirm", "exp19_two_stage"):
    run.log(f"  {k}: {DIRS.get(k, '없음')}")

def arm_at(d, name):
    p = os.path.join(d, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

def find_arm(name):
    for d in PARENTS:
        a = arm_at(d, name)
        if a is not None:
            return a
    return None

def seeds_of(c, cap=8):
    """그 구성에 실제로 존재하는 시드를 세어본다(실험19 실행 여부와 무관하게 동작)."""
    out = []
    for sd in range(cap):
        if all(find_arm(f"{c}_s{sd}_f{k}") is not None for k in range(K_FOLD)):
            out.append(sd)
    return out

In [ ]:
# CELL 2 — 라벨 + 【G0】 어휘·정합성·표본 확인 (신호는 안 읽는다)
import pandas as pd, subprocess

PTB = "/content/ptbxl"; os.makedirs(PTB, exist_ok=True)
d = os.path.join(PTB, "ptbxl_database.csv")
if not (os.path.exists(d) and os.path.getsize(d) > 0):
    subprocess.run(["wget", "-q", "-O", d,
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"])
df = pd.read_csv(d, index_col="ecg_id")

# arm 과 정렬을 맞추려면 실험15 캐시의 eid 순서를 그대로 써야 한다
CACHE = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"전량 캐시가 없습니다: {CACHE}")
z = np.load(CACHE, allow_pickle=True)
EID, FOLD10 = z["eid"], z["fold"]
CV = (FOLD10 - 1) % K_FOLD
run.log(f"레코드 {len(EID):,}건 (신호는 로드하지 않는다 — 학습 0회)")

dfa = df.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
counts = {c: int(sum(c in cs for cs in dfa.codes)) for c in sorted(vocab)}

# ── 【G0-a】 부위 어휘 (실험15~19 와 동일해야 arm 열이 맞는다)
assert_label_vocab(SITE_CANDIDATES, vocab, kind="MI 부위 코드",
                   counts=counts, min_count=NMIN)
SITES = [s for s in SITE_CANDIDATES if counts[s] >= NMIN]
NS = len(SITES)
S18 = json.load(open(os.path.join(DIRS["exp18_confirm"], "result.json"),
                     encoding="utf-8"))["sites"]
if SITES != S18:
    raise RuntimeError(f"부위 목록/순서가 실험18과 다릅니다: {SITES} vs {S18}")
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype(bool)
run.log(f"【G0-a】 부위 정합 ✅ {SITES}")

# ── 【G0-b】 감별질환 어휘 — **없는 코드는 조용히 빠지지 않고 로그에 남긴다**
WANT_G = MIMIC_Q + CONDUCT + EXTRA + ["NORM"]
missing = [g for g in WANT_G if g not in vocab]
if missing:
    run.log(f"【G0-b】 ⚠️ PTB-XL 에 없는 코드 {missing} — 분석에서 뺀다")
GROUPS = [g for g in WANT_G if g in vocab]
assert_label_vocab(GROUPS, vocab, kind="감별질환 코드", counts=counts, min_count=GMIN)
for nm, lst in (("P1 주가설", [P1_MIMIC]), ("모방군", MIMIC_Q), ("전도장애군", CONDUCT)):
    gone = [g for g in lst if g not in GROUPS]
    if gone:
        raise RuntimeError(f"{nm} 에 필요한 코드 {gone} 가 없습니다 — 채점 불가")

# ── MI 계열 제외 마스크 (깨끗한 모방만 남긴다)
mi_present = [c for c in MI_LIKE if c in vocab]
run.log(f"【G0-c】 MI 계열 제외 코드 {mi_present}"
        + (f" · 없는 것 {[c for c in MI_LIKE if c not in vocab]}" if
           any(c not in vocab for c in MI_LIKE) else ""))
HAS_MI = np.array([any(c in cs for c in mi_present) for cs in dfa.codes])
run.log(f"  MI 계열 보유 {int(HAS_MI.sum()):,}건 → 모방 검정에서 제외")

# ── 군 마스크: 그 소견이 있고 MI 는 없는 레코드
G_MASK, G_N = {}, {}
for g in GROUPS:
    m = np.array([g in cs for cs in dfa.codes]) & (~HAS_MI)
    G_MASK[g] = m; G_N[g] = int(m.sum())
run.log(f"\n【G0-d】 군별 표본 (MI 제외 후) — 채점 하한 n>={GMIN}")
for g in GROUPS:
    run.log(f"  {g:<8}{G_N[g]:>7,}{'' if G_N[g] >= GMIN else '   ⚠️ 표본 부족 → 채점 제외'}")
SCOREABLE = [g for g in GROUPS if G_N[g] >= GMIN]
for nm, lst in (("P1", [P1_MIMIC]), ("모방군", MIMIC_Q), ("전도장애군", CONDUCT)):
    ok = [g for g in lst if g in SCOREABLE]
    if not ok:
        raise RuntimeError(f"{nm} 이 전부 표본 부족입니다 — 채점 불가")
    if len(ok) < len(lst):
        run.log(f"  ⚠️ {nm} 중 표본 부족으로 빠지는 것: {[g for g in lst if g not in ok]}")

# ── 사용 가능한 arm 확인
SEEDS = {c: seeds_of(c) for c in CFGS}
run.log("\n【G0-e】 구성별 사용 가능 시드")
for c in CFGS:
    run.log(f"  {c:<14}{SEEDS[c]}")
CFGS = [c for c in CFGS if SEEDS[c]]
if REF_CFG not in CFGS:
    raise RuntimeError(f"기준 구성 {REF_CFG} 의 arm 이 없습니다")
if DEPLOY not in CFGS:
    run.log(f"  ⚠️ 배포 후보 {DEPLOY} arm 이 없다 — P-4 는 미결로 남는다(실험19 먼저)")

In [ ]:
# CELL 3 — OOF 적재 + 동작점(단일단계·2단계) 산출
def load_oof(c, sd):
    o = np.zeros((len(EID), NS), "float32")
    for k in range(K_FOLD):
        te = np.where(CV == k)[0]
        a = find_arm(f"{c}_s{sd}_f{k}")
        assert_arm_shape(a, len(te), name=f"{c}_s{sd}_f{k}")
        o[te] = a
    return o

P = {c: {sd: load_oof(c, sd) for sd in SEEDS[c]} for c in CFGS}
run.log(f"OOF 적재 완료 — " + " · ".join(f"{c} 시드{len(SEEDS[c])}개" for c in CFGS))

def split(k):
    """★ 실험15~19 와 동일한 분할(임계값을 검증 겹에서 잡기 위함)."""
    te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    return te, rest[:n_val], rest[n_val:]

def thr_at_sens(score, pos, t):
    p = score[pos]
    return float(np.quantile(p, 1.0 - t, method="lower")) if len(p) else -np.inf

YANY = Ymul.any(1)

def alarms(c, sd):
    """부위별 경보 불리언. 단일단계와 2단계를 **둘 다** 낸다.

    2단계는 실험19 의 설계를 그대로 쓴다 — 1단계 MI-any 게이트(민감도 SENS_L1)를
    통과한 것만 대상. 실험19 가 아직 안 돌았어도 여기서 독립적으로 계산되므로
    **이 노트북은 실험19 결과에 의존하지 않는다.**
    """
    s1 = P[c][sd].max(1)
    one = np.zeros((len(EID), NS), bool); two = np.zeros((len(EID), NS), bool)
    for j in range(NS):
        y = Ymul[:, j]
        for k in range(K_FOLD):
            te, va, _ = split(k)
            t1 = thr_at_sens(P[c][sd][va, j], y[va], SENS)
            one[te, j] = P[c][sd][te, j] >= t1
            tg = thr_at_sens(s1[va], YANY[va], SENS_L1)
            g_va = s1[va] >= tg
            sub = va[g_va]
            t2 = thr_at_sens(P[c][sd][sub, j], y[sub], SENS) if len(sub) else -np.inf
            two[te, j] = (s1[te] >= tg) & (P[c][sd][te, j] >= t2)
    return one, two

A1 = {c: {sd: None for sd in SEEDS[c]} for c in CFGS}
A2 = {c: {sd: None for sd in SEEDS[c]} for c in CFGS}
for c in CFGS:
    for sd in SEEDS[c]:
        A1[c][sd], A2[c][sd] = alarms(c, sd)
    run.log(f"  {c:<14} 동작점 산출 완료")

# 실측 민감도 — 전제 확인
for c in CFGS:
    sv = np.mean([[ (A1[c][sd][:, j] & Ymul[:, j]).sum() / max(Ymul[:, j].sum(), 1)
                    for j in range(NS)] for sd in SEEDS[c]])
    run.log(f"  {c:<14} 단일단계 실측 민감도 평균 {sv:.3f} (목표 {SENS})")

In [ ]:
# CELL 4 — 위양성 분해: FPR(부위 × 감별질환군)
from scipy import stats

def t_ci(vals, conf=0.95):
    v = np.asarray([x for x in vals if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2:
        return m, np.nan, np.nan, 0.0
    sd = float(v.std(ddof=1))
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * sd / np.sqrt(n))
    return m, m - h, m + h, sd

def fpr(alarm, j, mask):
    """부위 j 에 대해 mask 집단의 위양성률. **그 부위 양성인 레코드는 뺀다.**"""
    sel = mask & (~Ymul[:, j])
    return float(alarm[sel, j].mean()) if sel.sum() else np.nan

def table(c, A):
    """FPR[군][부위] = 시드별 리스트."""
    out = {g: {s: [fpr(A[c][sd], j, G_MASK[g]) for sd in SEEDS[c]]
               for j, s in enumerate(SITES)} for g in GROUPS}
    return out

FP1 = {c: table(c, A1) for c in CFGS}
FP2 = {c: table(c, A2) for c in CFGS}

N_NORM = {s: int((G_MASK["NORM"] & (~Ymul[:, j])).sum()) for j, s in enumerate(SITES)}

def ratio(c, g, s, FP):
    """군 g 의 부위 s 위양성 **배수**(NORM 대비) — 시드별.

    ★ 분모 가드: NORM 의 기대 위양성 건수가 MIN_FP 미만이면 배수가 폭발한다.
      그런 칸은 nan 으로 두고 배수 대신 **위험차**(diff)로 읽는다.
    """
    out = []
    for i in range(len(SEEDS[c])):
        den = FP[c]["NORM"][s][i]
        num = FP[c][g][s][i]
        out.append(num / den if (np.isfinite(den) and np.isfinite(num)
                                 and den * N_NORM[s] >= MIN_FP) else np.nan)
    return out

def diff(c, g, s, FP):
    """위험차 FPR(군) − FPR(NORM). 분모가 작아도 안 터진다 — 배수와 함께 본다."""
    return [FP[c][g][s][i] - FP[c]["NORM"][s][i] for i in range(len(SEEDS[c]))]

def _odds(x, eps=1e-6):
    x = min(max(float(x), eps), 1.0 - eps)
    return x / (1.0 - x)

def orat(c, g, s, FP):
    """**오즈비** — 판정에 쓰는 주 지표. 위험비와 달리 천장이 없다."""
    out = []
    for i in range(len(SEEDS[c])):
        den, num = FP[c]["NORM"][s][i], FP[c][g][s][i]
        out.append(_odds(num) / _odds(den)
                   if (np.isfinite(den) and np.isfinite(num)
                       and den * N_NORM[s] >= MIN_FP) else np.nan)
    return out

run.log("\n" + "=" * 124)
run.log(f"【위양성률】 {REF_CFG} · 단일단계 · 민감도 {SENS} · 시드 평균")
run.log("=" * 124)
run.log(f"  {'군':<8}{'n':>7}" + "".join(f"{s:>10}" for s in SITES) + f"{'macro배수':>11}")
for g in GROUPS:
    r = [np.nanmean(ratio(REF_CFG, g, s, FP1)) for s in SITES]
    mark = ""
    if g in MIMIC_Q: mark = "  ← 모방군"
    elif g in CONDUCT: mark = "  ← 전도장애(대조)"
    elif g == "NORM": mark = "  ← 기준"
    run.log(f"  {g:<8}{G_N[g]:>7,}"
            + "".join(f"{np.nanmean(FP1[REF_CFG][g][s]):>10.3f}" for s in SITES)
            + f"{np.nanmean(r):>11.2f}배{mark}"
            + ("" if G_N[g] >= GMIN else "  ⚠️표본부족"))

run.log(f"\n【배수 = FPR(군) / FPR(NORM)】 {REF_CFG} · 1.0 이면 정상과 같다")
run.log(f"  {'군':<8}" + "".join(f"{s:>10}" for s in SITES))
for g in [x for x in GROUPS if x != "NORM"]:
    run.log(f"  {g:<8}" + "".join(f"{np.nanmean(ratio(REF_CFG, g, s, FP1)):>10.2f}"
                                  for s in SITES))

run.log(f"\n【오즈비 OR】 {REF_CFG} · **판정에 쓰는 지표**(천장 없음)")
run.log(f"  {'군':<8}" + "".join(f"{s:>10}" for s in SITES) + f"{'macro':>9}")
for g in [x for x in GROUPS if x != "NORM"]:
    v = [np.nanmean(orat(REF_CFG, g, s, FP1)) for s in SITES]
    run.log(f"  {g:<8}" + "".join(f"{x:>10.2f}" for x in v) + f"{np.nanmean(v):>9.2f}")
cap = [1.0 / np.nanmean(FP1[REF_CFG]["NORM"][s]) if np.nanmean(FP1[REF_CFG]["NORM"][s]) else np.nan
       for s in SITES]
run.log(f"\n  ※ 위험비 천장(=1/FPR(NORM)): "
        + " ".join(f"{s} {c:.1f}" for s, c in zip(SITES, cap)))
run.log("     천장이 2 에 가까운 부위에서는 **위험비로는 '2배' 를 물을 수 없다** → OR 로 판정한다")

run.log(f"\n【위험차 = FPR(군) − FPR(NORM)】 분모가 작을 때도 안 터진다")
run.log(f"  {'군':<8}" + "".join(f"{s:>10}" for s in SITES))
for g in [x for x in GROUPS if x != "NORM"]:
    run.log(f"  {g:<8}" + "".join(f"{np.nanmean(diff(REF_CFG, g, s, FP1)):>+10.3f}"
                                  for s in SITES))
nan_cells = sum(1 for g in GROUPS for s in SITES
                if not np.isfinite(np.nanmean(ratio(REF_CFG, g, s, FP1))))
if nan_cells:
    run.log(f"\n  ⚠️ 배수를 못 낸 칸 {nan_cells}개 — NORM 기대 위양성 < {MIN_FP}건. "
            "그 칸은 위험차로 읽는다")
run.log(f"  NORM 표본(부위별): " + " ".join(f"{s} {N_NORM[s]:,}" for s in SITES))

In [ ]:
# CELL 5 — 사전등록 채점
run.log("\n" + "=" * 110)
run.log("【사전등록 채점】 시드 t-CI · 소집단은 레코드 부트스트랩도 함께 (넓은 쪽 채택)")
run.log("=" * 110)
V = {}

def boot_ratio_ci(c, g, s, A, B=BOOT):
    """레코드 부트스트랩 배수 CI — 시드 잡음이 아니라 **표본 잡음**을 잰다.
    소집단(WPW 등)에서는 이쪽이 지배적이므로 둘 중 넓은 CI 를 쓴다."""
    j = SITES.index(s)
    sd0 = SEEDS[c][0]
    a = A[c][sd0][:, j]
    gm = np.where(G_MASK[g] & (~Ymul[:, j]))[0]
    nm = np.where(G_MASK["NORM"] & (~Ymul[:, j]))[0]
    if len(gm) < 5 or len(nm) < 5:
        return np.nan, np.nan
    rs = np.random.RandomState(SEED0)
    out = []
    for _ in range(B):
        x = a[gm[rs.randint(0, len(gm), len(gm))]].mean()
        y = a[nm[rs.randint(0, len(nm), len(nm))]].mean()
        out.append(x / y if y > 0 else np.nan)
    out = np.array(out, float); out = out[np.isfinite(out)]
    return (float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))) \
        if len(out) else (np.nan, np.nan)

def widest(seed_ci, boot_ci):
    lo = np.nanmin([seed_ci[0], boot_ci[0]]); hi = np.nanmax([seed_ci[1], boot_ci[1]])
    return float(lo), float(hi)

# ── P-1 ★★ 주가설 (단일 검정) — 오즈비
r = orat(REF_CFG, P1_MIMIC, P1_SITE, FP1)
m, lo, hi, sd = t_ci(r)
blo, bhi = boot_ratio_ci(REF_CFG, P1_MIMIC, P1_SITE, A1)
LO, HI = widest((lo, hi), (blo, bhi))
V["P-1"] = decide(LO, HI, OR_THR, ">")
run.log(f"\n  P-1 {P1_MIMIC} → {P1_SITE} 위양성 **오즈비** = {m:.2f}")
run.log(f"      시드 t-CI [{lo:.2f}, {hi:.2f}] · 부트스트랩 [{blo:.2f}, {bhi:.2f}] "
        f"→ 채택 [{LO:.2f}, {HI:.2f}] vs {OR_THR} → {MARK[V['P-1']]}")
run.log(f"      FPR: {P1_MIMIC} {np.nanmean(FP1[REF_CFG][P1_MIMIC][P1_SITE]):.3f} vs "
        f"NORM {np.nanmean(FP1[REF_CFG]['NORM'][P1_SITE]):.3f} "
        f"· 위험차 {np.nanmean(diff(REF_CFG, P1_MIMIC, P1_SITE, FP1)):+.3f} "
        f"· 위험비 {np.nanmean(ratio(REF_CFG, P1_MIMIC, P1_SITE, FP1)):.2f} "
        f"(n={G_N[P1_MIMIC]:,})")

# ── P-2 모방군 macro
MQ = [g for g in MIMIC_Q if g in SCOREABLE]
CD = [g for g in CONDUCT if g in SCOREABLE]
def macro_or(c, lst, FP, per_site=None):
    """★ 부위를 **기전이 예측하는 곳으로 제한**한다. 7부위 평균은 희석된다.

    per_site 가 주어지면 모방별 표적 부위에서만, 없으면 TARGET_SITES 합집합에서.
    대조군(전도장애)은 예측 부위가 없으므로 **같은 합집합**에서 재야 공정하다.
    """
    pairs = ([(g, s) for g in lst for s in per_site.get(g, []) if s in SITES]
             if per_site else
             [(g, s) for g in lst for s in TARGET_SITES if s in SITES])
    return [float(np.nanmean([orat(c, g, s, FP)[i] for g, s in pairs]))
            for i in range(len(SEEDS[c]))]

def macro_or_all(c, lst, FP):
    """7부위 전체 macro — **탐색 보고용**(판정에는 안 쓴다)."""
    return [float(np.nanmean([[orat(c, g, s, FP)[i] for s in SITES] for g in lst]))
            for i in range(len(SEEDS[c]))]

m2, lo2, hi2, sd2 = t_ci(macro_or(REF_CFG, MQ, FP1, MIMIC_SITES))
V["P-2"] = decide(lo2, hi2, OR_THR, ">")
run.log(f"\n  P-2 모방군 macro OR = {m2:.2f} [{lo2:.2f}, {hi2:.2f}] vs {OR_THR} "
        f"→ {MARK[V['P-2']]}")
run.log("      표적 부위(사전지정): " + " · ".join(f"{g}→{MIMIC_SITES[g]}" for g in MQ))
run.log(f"      (참고) 7부위 전체 macro = "
        f"{np.nanmean(macro_or_all(REF_CFG, MQ, FP1)):.2f} — 부위 특이 기전이 희석된 값")

# ── P-3 ★★ 특이성 대조 — 여기가 핵심이다
mq = macro_or(REF_CFG, MQ, FP1, MIMIC_SITES); cd = macro_or(REF_CFG, CD, FP1)
m3, lo3, hi3, sd3 = t_ci([mq[i] - cd[i] for i in range(len(mq))])
V["P-3"] = decide(lo3, hi3, P3_MARGIN, ">")
run.log(f"\n  P-3 특이성 대조 — 모방군 OR {np.nanmean(mq):.2f} vs "
        f"전도장애군 {CD} OR {np.nanmean(cd):.2f}")
run.log(f"      차 = {m3:+.2f} [{lo3:+.2f}, {hi3:+.2f}] vs **최소 유의미 차 {P3_MARGIN}** "
        f"→ {MARK[V['P-3']]}")
run.log(f"      ※ 문턱을 0 이 아니라 {P3_MARGIN} 로 둔 이유: 표본 21,799건에 시드 3~6개면 "
        "SD 가 극히 작아")
run.log("        +0.01 짜리 차이도 CI 가 0 을 벗어난다(픽스처에서 귀무가 통과했다). "
        "최소 유의미 차를 사전에 못 박는다")
run.log("      " + ("→ Q파 기전에서 선택적으로 크다. P-1·P-2 가 의미를 갖는다"
                    if V["P-3"] else
                    "→ ⚠️ 전도장애군과 구별되지 않는다. **'비정상 심전도는 다 위양성이 많다'** "
                    "는 사소한 설명을 못 배제하므로 P-1·P-2 를 그대로 읽으면 안 된다"))

# ── P-4 ★★ 배포 후보 검증
if DEPLOY in CFGS:
    dq = macro_or(DEPLOY, MQ, FP1, MIMIC_SITES)
    n_pair = min(len(dq), len(mq))
    m4, lo4, hi4, sd4 = t_ci([dq[i] - mq[i] for i in range(n_pair)])
    V["P-4"] = decide(lo4, hi4, P4_SLACK, "<")
    run.log(f"\n  P-4 배포 후보 — {DEPLOY} OR {np.nanmean(dq):.2f} vs "
            f"{REF_CFG} OR {np.nanmean(mq):.2f}")
    run.log(f"      차 = {m4:+.2f} [{lo4:+.2f}, {hi4:+.2f}] vs 여유 {P4_SLACK} "
            f"→ {MARK[V['P-4']]}")
    run.log("      " + ("→ 유도를 줄여도 감별 능력이 안 무너진다. "
                        "'5전극 = 12유도' 를 감별 축에서도 지지한다"
                        if V["P-4"] else
                        "→ ⚠️ **'5전극 = 12유도' 헤드라인에 단서를 달아야 한다** — "
                        "AUROC 는 같은데 감별에서 진다"))
else:
    V["P-4"] = None; m4 = float("nan")
    run.log(f"\n  P-4 → {DEPLOY} arm 이 없어 미결 (실험19 를 먼저 돌린다)")

# ── P-5 2단계 동작점
mq2 = macro_or(REF_CFG, MQ, FP2, MIMIC_SITES)
m5, lo5, hi5, sd5 = t_ci([mq2[i] - mq[i] for i in range(len(mq))])
V["P-5"] = decide(lo5, hi5, 0.0, "<")
run.log(f"\n  P-5 2단계 동작점 — 모방군 OR 단일 {np.nanmean(mq):.2f} → "
        f"2단계 {np.nanmean(mq2):.2f}")
run.log(f"      차 = {m5:+.2f} [{lo5:+.2f}, {hi5:+.2f}] → {MARK[V['P-5']]}")

run.log("\n  【효과 vs 시드 잡음】 규약 ②")
for nm_, mm, ss in (("P-3 특이성", m3, sd3), ("P-5 2단계", m5, sd5)):
    r_ = abs(mm) / ss if ss > 0 else float("inf")
    run.log(f"      {nm_:<12} |효과| {abs(mm):.3f} / SD {ss:.3f} = {r_:.2f}배"
            + ("" if r_ >= 1 else "  ⚠️ 잡음 이하 — 검출 불가로 종결"))

run.log("\n" + "=" * 110)
for k in ("P-1", "P-2", "P-3", "P-4", "P-5"):
    run.log(f"  {k}: {MARK[V.get(k)]}")
run.log("=" * 110)

# ── 라벨 트리 우선순위 (이 실험의 산출물)
run.log("\n【라벨 트리 우선순위】 위양성 배수가 큰 순 — 어느 감별질환부터 라벨에 넣을 것인가")
rank = sorted([(g, float(np.nanmean([np.nanmean(orat(REF_CFG, g, s, FP1))
                                     for s in SITES])), G_N[g])
               for g in SCOREABLE if g != "NORM"], key=lambda x: -x[1])
for i, (g, v, n) in enumerate(rank, 1):
    run.log(f"  {i:>2}. {g:<8} macro OR {v:>5.2f}  (n={n:,})"
            + ("  ★ 사전지정 모방군" if g in MIMIC_Q else
               "  (전도장애 대조)" if g in CONDUCT else ""))
run.log("  → 상위 군부터 2단계 라벨 트리에 넣는다(추측이 아니라 측정으로 정한다)")

In [ ]:
# CELL 6 — 그림
import matplotlib.pyplot as plt
show = [g for g in SCOREABLE if g != "NORM"]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

M = np.array([[np.nanmean(orat(REF_CFG, g, s, FP1)) for s in SITES] for g in show])
im = ax[0].imshow(M, cmap="Reds", vmin=1.0, vmax=max(2.0, np.nanmax(M)), aspect="auto")
ax[0].set_xticks(range(len(SITES))); ax[0].set_xticklabels(SITES, rotation=45, ha="right")
ax[0].set_yticks(range(len(show)))
ax[0].set_yticklabels([g + (" ★" if g in MIMIC_Q else "") for g in show])
for i in range(len(show)):
    for j in range(len(SITES)):
        if np.isfinite(M[i, j]):
            ax[0].text(j, i, f"{M[i, j]:.1f}", ha="center", va="center", fontsize=7,
                       color="white" if M[i, j] > 2.2 else "black")
ax[0].set_title(f"위양성 오즈비 (NORM=1) · {REF_CFG}")
plt.colorbar(im, ax=ax[0], fraction=.04)

col = ["tab:red" if g in MIMIC_Q else "tab:blue" if g in CONDUCT else "0.6" for g in show]
mv = [float(np.nanmean([np.nanmean(orat(REF_CFG, g, s, FP1)) for s in SITES]))
      for g in show]
o = np.argsort(mv)
ax[1].barh(range(len(show)), [mv[i] for i in o], color=[col[i] for i in o])
ax[1].axvline(1.0, color="k", lw=1); ax[1].axvline(OR_THR, color="green", ls="--", lw=1.3)
ax[1].set_yticks(range(len(show))); ax[1].set_yticklabels([show[i] for i in o])
ax[1].set_xlabel("macro 위양성 오즈비")
ax[1].set_title(f"빨강=모방군 · 파랑=전도장애(대조) · P-3 {MARK[V['P-3']]}")

if DEPLOY in CFGS:
    a = [float(np.nanmean([np.nanmean(orat(REF_CFG, g, s, FP1)) for s in SITES]))
         for g in MQ]
    b = [float(np.nanmean([np.nanmean(orat(DEPLOY, g, s, FP1)) for s in SITES]))
         for g in MQ]
    x = np.arange(len(MQ)); w = .38
    ax[2].bar(x - w/2, a, w, label=f"{REF_CFG} (10전극)", color="0.5")
    ax[2].bar(x + w/2, b, w, label=f"{DEPLOY} (5전극)", color="tab:orange")
    ax[2].axhline(1.0, color="k", lw=1)
    ax[2].set_xticks(x); ax[2].set_xticklabels(MQ)
    ax[2].set_ylabel("macro 위양성 오즈비"); ax[2].legend(fontsize=8)
    ax[2].set_title(f"유도를 줄이면 감별이 무너지나 · P-4 {MARK[V['P-4']]}")
else:
    ax[2].text(.5, .5, f"{DEPLOY} arm 없음\n(실험19 먼저)", ha="center", va="center")
    ax[2].axis("off")

plt.tight_layout(); run.save_fig("mimic_fp", fig); plt.show()

In [ ]:
# CELL 7 — 결과 저장
res = {
    "week": 2, "exp_id": "exp22_mimic_fp", "quest": "ailab-2026-0015",
    "task": "경색을 흉내내는 소견별 위양성 분해 — 임상 특이도의 진짜 취약점",
    "split": "inter", "step": "exp22-mimic-false-positives",
    "metric": f"fp_odds_ratio_{P1_MIMIC}_{P1_SITE}",
    "value": round(float(np.nanmean(orat(REF_CFG, P1_MIMIC, P1_SITE, FP1))), 3),
    "passed": bool(V.get("P-1") is True and V.get("P-3") is True),
    "date": time.strftime("%Y-%m-%d"),
    "training_runs": 0, "k_fold": K_FOLD, "sens": SENS,
    "sites": SITES, "configs": CFGS, "seeds": {c: SEEDS[c] for c in CFGS},
    "groups": GROUPS, "scoreable": SCOREABLE, "group_n": G_N,
    "mimic_q": MQ, "conduct": CD,
    "mimic_sites": MIMIC_SITES, "target_sites": TARGET_SITES,
    "p2_macro_all_sites": round(float(np.nanmean(macro_or_all(REF_CFG, MQ, FP1))), 3),
    "P-1": V.get("P-1"), "P-2": V.get("P-2"), "P-3": V.get("P-3"),
    "P-4": V.get("P-4"), "P-5": V.get("P-5"),
    "metric_scale": "odds ratio (위험비는 천장 1/FPR(NORM) 때문에 판정에 부적합)",
    "or_thr": OR_THR, "p3_margin": P3_MARGIN,
    "p1_or": round(float(m), 3), "p1_ci": [round(LO, 3), round(HI, 3)],
    "p2_macro": round(float(m2), 3), "p2_ci": [round(lo2, 3), round(hi2, 3)],
    "p3_diff": round(float(m3), 3), "p3_ci": [round(lo3, 3), round(hi3, 3)],
    "p4_diff": (round(float(m4), 3) if DEPLOY in CFGS else None),
    "p5_diff": round(float(m5), 3), "p5_ci": [round(lo5, 3), round(hi5, 3)],
    "fpr_or": {g: {s: round(float(np.nanmean(orat(REF_CFG, g, s, FP1))), 3)
                   for s in SITES} for g in GROUPS},
    "fpr_ratio": {g: {s: round(float(np.nanmean(ratio(REF_CFG, g, s, FP1))), 3)
                      for s in SITES} for g in GROUPS},
    "fpr_raw": {g: {s: round(float(np.nanmean(FP1[REF_CFG][g][s])), 4) for s in SITES}
                for g in GROUPS},
    "fpr_diff": {g: {s: round(float(np.nanmean(diff(REF_CFG, g, s, FP1))), 4)
                     for s in SITES} for g in GROUPS},
    "n_norm_by_site": N_NORM, "min_fp_guard": MIN_FP,
    "label_tree_priority": [{"code": g, "macro_or": round(v, 3), "n": n}
                            for g, v, n in rank],
    "verdict": " · ".join(f"{k} {MARK[V.get(k)]}" for k in
                          ("P-1", "P-2", "P-3", "P-4", "P-5")),
}
res["summary"] = (f"{P1_MIMIC}→{P1_SITE} OR {m:.2f} · 모방군 macro OR {m2:.2f} · "
                  f"특이성 대조 {m3:+.2f} · " + res["verdict"])
run.save_json("result.json", res)
run.log("\n" + json.dumps({k: res[k] for k in
                           ("metric", "value", "passed", "P-1", "P-2", "P-3", "P-4",
                            "P-5", "label_tree_priority", "summary")},
                          ensure_ascii=False, indent=2))
run.finish(res)